# Uniform RNG distribution

We need to test if the distribution is normal or non-normal to let us be confident in the RNG seed finding as a probabalistic comparison.

In [1]:
from seedler import Planter, Sprout, Fire, Nursery
import pandas as pd
from lets_plot import *
LetsPlot.setup_html()

We will use a sampling range of 100, and run 10-million tests to improve out accuracy and certainty.

In [5]:
test_range = 100
test_runs = 10_000_000

Our lab simply generates a single bud within our random range.

In [6]:
class MyLab(Planter):
    def plant(self, sprout: Sprout):
        sprout.add_bud(sprout.growth(0,test_range - 1))

Now, we can simulate the lab across out `test_runs`, and convert the planter results into a pandas dataframe.

In [7]:
import numpy as np

print("INITIALIZING SIM")
lab = MyLab()

print("SIMULATING")
# Ensure your Rust find_seeds returns the list of (seed, dict)
planter = lab.find_seeds(maximum=test_runs)

print("AGGREGATING DATA (NumPy Optimized)")
# Initialize a NumPy array (highly optimized C-array)
# test_range must match the possible number of keys
total_counts = np.zeros(test_range, dtype=np.int64)

# We still iterate once over the 'planter' list, 
# but we aggregate keys in bulk
for _, data_dict in planter:
    if not data_dict:
        continue
    # Extract keys and values as arrays
    keys = np.fromiter(data_dict.keys(), dtype=np.int32)
    vals = np.fromiter(data_dict.values(), dtype=np.int32)
    # Vectorized addition
    total_counts[keys] += vals

print("CONVERTING TO PANDAS")
# Create the DataFrame directly from the NumPy array
df = pd.DataFrame({
    "Key": np.arange(test_range),
    "Count": total_counts
})

print("PROCESS COMPLETED")

df

INITIALIZING SIM
SIMULATING
AGGREGATING DATA (NumPy Optimized)
CONVERTING TO PANDAS
PROCESS COMPLETED


,Key,Count
0,0,99742
1,1,100164
2,2,99676
3,3,100374
4,4,99840
...,...,...
95,95,100050
96,96,100135
97,97,99423
98,98,100067


Here we plot our results for visual verification, where it appears to be a normal distribution.

In [8]:
ggplot(
    df,
    aes(x="Key", y="Count")
) + geom_bar(stat="identity")

Then we can mathematically compare the probability of our distribution being truly normal.

In [10]:
exp_mean = (test_range - 1) / 2.0
exp_std = ((test_range**2 - 1) / 12) ** 0.5
exp_mean_error = exp_std / (df["Count"].sum() ** 0.5)

total_samples = int(df["Count"].sum())

actual_mean = (df["Key"] * df["Count"]).sum() / total_samples

weighted_variance = (df["Count"] * (df["Key"] - actual_mean)**2).sum() / total_samples
actual_std = weighted_variance ** 0.5

actual_mean_error = abs(actual_mean - exp_mean)

print(f"Total Samples Run:  {total_samples:,}")
print("-" * 40)
print(f"Expected Mean:      {exp_mean:>10.3f} | Actual Weighted Mean: {actual_mean:>10.3f}")
print(f"Expected STD:       {exp_std:>10.3f} | Actual Weighted STD:  {actual_std:>10.3f}")
print(f"Expected Error:     {exp_mean_error:>10.3f} | Actual Mean Error:    {actual_mean_error:>10.3f}")

Total Samples Run:  10,000,000
----------------------------------------
Expected Mean:          49.500 | Actual Weighted Mean:     49.490
Expected STD:           28.866 | Actual Weighted STD:      28.864
Expected Error:          0.009 | Actual Mean Error:         0.010


Over a simulation with `test_range=100` and `test_runs=10_000_000`, we are very close to all expected values. The deviation is close to 1 standard deviation, and between the 1st and 2nd deviation, suggesting that the distribution of the backend rust RNG is either a truly normal distribution, or very close to it.

This allows us to have full confidence in the distribution of our seeds as we run more complex simulations.